# Sentinel-2 workflow (Copernicus)
Same steps as `Biomass_Downloader.ipynb`, for Sentinel-2 Level-2A products (surface reflectance, 100 x 100 km tiles). The logic lives in the `biomass` package (`src/biomass`). Install once with `pip install -e ".[geo,dev]"`, edit only the configuration cell and run top to bottom.

### Copernicus account
Searching and the thumbnails need no account. The files themselves are served by the **Copernicus Data Space**, which needs its own free account (your MAAP `token.txt` does not work there): create one at https://dataspace.copernicus.eu.

**Recommended: save your login in `copernicus.txt`**, so you are never asked for it. Create a plain text file with that name in the project folder (the same folder as `token.txt`), with exactly two lines:

```
your.email@example.com
your-password
```

- Line 1: the e-mail of your Copernicus account. Line 2: its password. Nothing else.
- The file is listed in `.gitignore`, so git will not upload it. Never write the password in a notebook cell: notebooks are committed.

**Without the file**, the first download asks for the e-mail and password:
- They are kept in memory, so later downloads in the same session do not ask again.
- They are forgotten when the kernel restarts or when the configuration cell is run again; the next download then asks again.
- A wrong password is forgotten at once, so you are simply asked again.
- Nothing you type there is saved to disk.

Behind the scenes, the code logs in once and receives a temporary access key that it renews by itself while the notebook runs; you never have to log in again during a session.

In [ ]:
from pathlib import Path
from biomass import (SentinelClient, SearchQuery, Downloader, sentinel_items_table, assets_table,
                     footprint_map, plot_optical, optical_map)

# --- Configuration ---
QUERY = SearchQuery(
    collection="sentinel-2-l2a",
    product_type="S2MSI2A",
    date_start="2026-01-01",
    date_end="2026-07-01",
    bbox=[-60.6, -3.4, -59.8, -2.8],           # [min_lon, min_lat, max_lon, max_lat]
    aoi_file=None,                             # e.g. "my_area.geojson" (overrides bbox)
    max_cloud_cover=20,                        # percent of the tile; None for any
    extra_filters={},                          # e.g. {"grid:code": "MGRS-20MQB"} for one tile
    max_items=10,                              # results come newest first
)
ASSETS_TO_DOWNLOAD = ["TCI_10m", "B04_10m", "B08_10m"]   # true colour, red, near infrared (10 m)
OUTPUT_DIR = Path("../data/sentinel2")

client = SentinelClient()

## Explore

In [ ]:
client.collections()

In [ ]:
client.queryables(QUERY.collection)

## Search
Tiles at the edge of an orbit can be almost empty: check `file:size` in `assets_table` before downloading.

In [ ]:
items = client.search(QUERY)
sentinel_items_table(items)

In [ ]:
footprint_map(items, QUERY)

## Download
The thumbnails are public. The product files need the Copernicus account described at the top: with `copernicus.txt` in place they download straight away, otherwise the first one asks for the e-mail and password.

In [ ]:
assets_table(items[0])

In [ ]:
thumbnails = Downloader(None, OUTPUT_DIR).download(items, ["thumbnail"])   # public

In [ ]:
files = Downloader(client.auth, OUTPUT_DIR).download(items[:1], ASSETS_TO_DOWNLOAD)   # test first with one

## Visualize
`plot_optical` shows one file on its UTM grid: the true colour image (TCI) in colour, a band in grey as surface reflectance. `optical_map` lays files over a map.

In [ ]:
tci = sorted(OUTPUT_DIR.rglob("*_TCI_10m.jp2"))
plot_optical(tci[0]);

In [ ]:
plot_optical(next(OUTPUT_DIR.rglob("*_B08_10m.jp2")));

In [ ]:
optical_map(tci)   # one layer per file; toggle them with the control in the top-right corner